In [1]:
import pandas as pd
import numpy as np
import sklearn
sklearn.set_config(transform_output="pandas")

from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from scipy.stats import uniform, loguniform
import warnings
warnings.filterwarnings('ignore')

In [2]:
rental = pd.read_csv('./data/data.csv')
rental.head()

,Property_id,Offer,URL,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent
0,6046,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3+ Maid,4,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0
1,2240,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0
2,6248,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,studio,1,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0
3,7177,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0
4,7842,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5+ Maid,5,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0


In [3]:
rental_copy = rental.copy()

In [4]:
rental_copy = rental_copy.dropna(subset=['rent'])

In [5]:
rental_copy = rental_copy.drop(columns=['Offer','URL'])

In [6]:
rental_copy['has_maid'] = rental_copy['Beds'].str.contains('Maid', case=False, na=False).astype(int)

In [7]:
rental_copy['Beds'] = rental_copy['Beds'].str.replace('Maid','',case=False)
rental_copy['Beds'] = rental_copy['Beds'].str.replace('+','',case=False)
rental_copy['Beds'] = rental_copy['Beds'].str.strip()
rental_copy['Beds'] = rental_copy['Beds'].replace('studio', '0')
rental_copy['Beds'] = pd.to_numeric(rental_copy['Beds'])

In [8]:
rental_copy['Baths'] = rental_copy['Baths'].str.replace('+','')
rental_copy['Baths'] = rental_copy['Baths'].replace('none', np.nan)
rental_copy['Baths'] = pd.to_numeric(rental_copy['Baths'])

In [9]:
rental_copy = rental_copy.dropna(subset=['Baths'])

In [10]:
rental_copy['Size_sqm'] = rental_copy['Size'].str.split('/').str[-1]
rental_copy['Size_sqm'] = rental_copy['Size_sqm'].str.replace('sqm', '', case=False)
rental_copy['Size_sqm'] = rental_copy['Size_sqm'].str.replace(',', '')
rental_copy['Size_sqm'] = pd.to_numeric(rental_copy['Size_sqm'], errors='coerce')
rental_copy = rental_copy.drop(columns=['Size'])

In [11]:
rental_copy['is_inclusive'] = (rental_copy['Include_w_e'] == 'Inclusive').astype(int)
rental_copy = rental_copy.drop(columns=['Include_w_e'])

In [12]:
rental_copy.info()

<class 'pandas.DataFrame'>
Index: 10574 entries, 0 to 10577
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Property_id        10574 non-null  int64  
 1   Property_type      10574 non-null  str    
 2   Title              10574 non-null  str    
 3   Area               10574 non-null  str    
 4   Governorate        10574 non-null  str    
 5   Beds               10574 non-null  int64  
 6   Baths              10574 non-null  float64
 7   Availability_date  10058 non-null  str    
 8   Agent_name         10574 non-null  str    
 9   Agency             10574 non-null  str    
 10  Amenities          10269 non-null  float64
 11  rent               10574 non-null  float64
 12  has_maid           10574 non-null  int64  
 13  Size_sqm           10574 non-null  int64  
 14  is_inclusive       10574 non-null  int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 2.6 MB


In [13]:
rental_copy.describe()

,Property_id,Beds,Baths,Amenities,rent,has_maid,Size_sqm,is_inclusive
count,10574.000000,10574.000000,10574.000000,10269.000000,10574.000000,10574.000000,10574.000000,10574.000000
mean,7060.277284,2.202478,2.663703,10.321258,718.784944,0.210800,171.275392,0.612446
std,4080.050950,1.206382,1.290132,5.093565,5318.277795,0.407896,204.971659,0.487215
min,2.000000,0.000000,1.000000,1.000000,50.000000,0.000000,1.000000,0.000000
25%,3522.500000,1.000000,2.000000,6.000000,330.000000,0.000000,90.000000,0.000000
50%,7023.500000,2.000000,2.000000,11.000000,450.000000,0.000000,125.000000,1.000000
75%,10605.750000,3.000000,3.000000,14.000000,700.000000,0.000000,180.000000,1.000000
max,14103.000000,7.000000,7.000000,25.000000,400000.000000,1.000000,13742.000000,1.000000


In [14]:
rental_copy['Title'] = rental_copy['Title'].fillna('')
title_lower = rental_copy['Title'].str.lower()

rental_copy['is_furnished'] = title_lower.str.contains('furnish', na=False).astype(int)
rental_copy['is_luxury'] = title_lower.str.contains('luxury|luxurious', na=False).astype(int)
rental_copy['has_pool'] = title_lower.str.contains('pool', na=False).astype(int)
rental_copy['has_sea_view'] = title_lower.str.contains('sea view|sea-view|seaview', na=False).astype(int)
rental_copy['is_spacious'] = title_lower.str.contains('spacious|large', na=False).astype(int)

rental_copy = rental_copy.drop(columns=['Title'])

In [15]:
rental_copy

,Property_id,Property_type,Area,Governorate,Beds,Baths,Availability_date,Agent_name,Agency,Amenities,rent,has_maid,Size_sqm,is_inclusive,is_furnished,is_luxury,has_pool,has_sea_view,is_spacious
0,6046,Villa,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3,4.0,9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0,1,220,1,0,1,0,0,0
1,2240,Villa,Diyar Al Muharraq,Muharraq Governorate,5,6.0,16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0,0,300,0,0,1,0,0,0
2,6248,Apartment,Al Juffair,Capital Governorate,0,1.0,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0,0,45,1,0,0,0,0,0
3,7177,Apartment,Jidhafs,Northern Governorate,3,3.0,6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0,0,140,0,0,0,0,0,0
4,7842,Villa,Al Juffair,Capital Governorate,5,5.0,30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0,1,450,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10573,5192,Apartment,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,2,3.0,29 Aug 2025,Al Ward Real Estate,AlWard Real Estate,17.0,550.0,0,135,1,1,0,0,0,0
10574,13419,Apartment,Al Juffair,Capital Governorate,2,2.0,NaN,Muhammad,Al Waleed Homes,8.0,300.0,0,120,0,1,0,0,0,0
10575,5391,Apartment,Seef,Capital Governorate,1,2.0,27 Aug 2025,Bilal Mohamed,River West Properties,11.0,650.0,0,91,1,0,0,0,1,0
10576,861,Villa,"Zinj, Manama",Capital Governorate,3,4.0,24 Sep 2025,Shaji Kottathazham,AIM REAL ESTATE,11.0,950.0,1,400,0,0,0,1,0,0


In [16]:
rental_copy.isnull().sum()

Property_id            0
Property_type          0
Area                   0
Governorate            0
Beds                   0
Baths                  0
Availability_date    516
Agent_name             0
Agency                 0
Amenities            305
rent                   0
has_maid               0
Size_sqm               0
is_inclusive           0
is_furnished           0
is_luxury              0
has_pool               0
has_sea_view           0
is_spacious            0
dtype: int64

In [17]:
rental_copy = rental_copy.drop(columns=['Availability_date'])

In [ ]:
rental_copy['Amenities'] = rental_copy['Amenities'].fillna(rental_copy['Amenities'].median())
print("Amenities nulls:", rental_copy['Amenities'].isnull().sum())